# 12 Promotion analysis from the shared dataset

**Questions:** What share of purchases carries a monetary discount, how deep are those discounts, and which code-based analyses remain unresolved?

Run 04 first. This notebook uses canonical corrected order and line amounts. The consumer purchase view includes positive purchases across platforms and excludes configured special accounts. A missing code does not imply no discount. Coupon text is not guessed from Excel date cells, and campaign families are not inferred here.

The figures describe recorded purchasing and discount amounts. They do not measure incremental sales, margin, acquisition cost or promotion profitability.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # run from the notebooks folder

import pandas as pd
from IPython.display import display
from weavve.clean import load_dataset

dataset = load_dataset()
print("Shared dataset ID:", dataset.manifest["dataset_id"])
print("Local dataset:", dataset.path)
print("Transaction cutoff:", dataset.manifest["transaction_cutoff"])
print("Review flags:", len(dataset.manifest["warnings"]))
pd.set_option("display.max_columns", 20)


## Monetary discount incidence and depth

A discounted purchase has corrected signed discount greater than zero. Depth is total discount divided by total net sales plus discount, using the same purchase population. This ratio is an exported-amount proxy; it is not the average advertised offer. Negative signed discounts are counted for review rather than silently clipped.


In [ ]:
orders = dataset.frame("orders")
purchases = orders.loc[orders["is_purchase"] & ~orders["is_special_account"]].copy()
purchases["discounted"] = purchases["discount"].map(lambda v: v > 0)
purchases["negative_signed_discount"] = purchases["discount"].map(lambda v: v < 0)
purchases["order_year"] = purchases["order_date"].map(lambda d: d.year)
summary = purchases.groupby(["order_year", "channel_detail"], dropna=False).agg(
    purchases=("order_id", "size"), discounted_purchases=("discounted", "sum"),
    negative_signed_discount_orders=("negative_signed_discount", "sum"),
    net_sales=("net", "sum"), discount=("discount", "sum"),
)
summary["discounted_purchase_pct"] = 100 * summary["discounted_purchases"] / summary["purchases"]
summary["discount_depth_pct"] = [100 * float(d / (n + d)) if n + d > 0 else None for n, d in zip(summary["net_sales"], summary["discount"])]
for column in ["net_sales", "discount"]:
    summary[column] = summary[column].map(float)
display(summary.round(2))


## Date-valued codes remain unresolved

Excel stored some source discount-code cells as dates. The shared flag preserves that fact while leaving the source cell untouched. These counts are line-level exposure, and one affected order can appear on several lines. Resolve the original code from an authoritative source before using it for exact attribution or campaign matching.


In [ ]:
lines = dataset.frame("order_lines")
date_codes = lines.loc[lines["discount_code_is_date"]].copy()
date_codes["order_year"] = date_codes["order_date"].map(lambda d: d.year)
display(date_codes.groupby("order_year").agg(
    retained_date_code_lines=("source_row", "size"), affected_orders=("order_id", "nunique"),
))


## Issued-code snapshot status

File E records issued discounts. Unused entries can arise from automated per-person issuance; their share does not demonstrate failed campaigns. The usage field is a source snapshot and does not provide a matched event-time denominator for the transaction period. No code names or customer identifiers are displayed.


In [ ]:
discounts = dataset.frame("discounts")
usage = discounts.groupby("clean_usage_observed", dropna=False).agg(issued_discount_rows=("source_row", "size"))
display(usage.rename(index={True: "usage recorded", False: "no usage recorded"}))
print("Rows with unknown usage:", int(discounts["clean_usage_observed"].isna().sum()))
